In [9]:
# Getting started with LangChain
# Simple LLM calls with streaming
# Dynamic Prompt Template ( translation app)
# Building chains ( story generator with analysis)
# Conversational Q&A assistant with memory
# Tool integration ( calculator & weather)
import langchain
import os
from dotenv import load_dotenv
load_dotenv()

#os.getenv("OPENAI_API_KEY")



True

In [ ]:
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

### Example 1: Simple LLM call with streaming


In [11]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage
# initialize model
model = init_chat_model(
    "groq:qwen/qwen3.8-27b"
)
model



ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.9', 'langchain': '1.4.4'}}, client=<groq.resources.chat.completions.Completions object at 0x107caead0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x107cbc710>, model_name='qwen/qwen3.8-27b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [12]:
# Create message
messages = [
        SystemMessage("You are a helpful assistant that can answer questions."),
        HumanMessage("What is the capital of France?")
    ]


In [13]:
response = model.invoke(messages)
response

AIMessage(content='The capital of France is **Paris**.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 34, 'total_tokens': 43, 'completion_time': 0.017258541, 'completion_tokens_details': None, 'prompt_time': 0.002143689, 'prompt_tokens_details': None, 'queue_time': 0.04347403, 'total_time': 0.01940223}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_a1293f40b5', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a12fe7-86a9-7802-bff2-32792b78a5e8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 34, 'output_tokens': 9, 'total_tokens': 43})

In [14]:
model.invoke([HumanMessage("What is machine learning?")])


AIMessage(content='**Machine Learning (ML)** is a subfield of **Artificial Intelligence (AI)** that focuses on building systems that learn from data. Instead of following explicit, hard-coded instructions, a machine learning system uses algorithms to analyze data, identify patterns, and make decisions or predictions that improve over time as it is exposed to more information.\n\n### Key Characteristics\n\n1.  **Data-Driven**: ML models are trained on large datasets. The quality and quantity of data directly impact the model\'s performance.\n2. **Pattern Recognition**: The core task is to find underlying structures or relationships in the data that can be used to generalize to new, unseen data.\n3.  **Iterative Improvement**: Unlike traditional programming, where you write rules for every scenario, ML models refine their internal parameters (weights) through repeated exposure to data, reducing errors.\n\n### How It Works (Simplified)\n\n1.  **Data Collection**: Gather relevant data (e.g

In [15]:
# streaming example
for chunk in model.stream(messages):
    print(chunk.content, end="", flush=True)


The capital of France is **Paris**.

## Dynamic Prompt Template

In [16]:
from langchain_core.prompts import ChatPromptTemplate
# Create translation app
source_language = "French"
target_language = "Hindi"
text = "I love programming in Python"
translation_template = ChatPromptTemplate.from_messages(
    [
        ("system", f"You are a professional translator. Translate the following {text} from {source_language} to {target_language}. Maintain the tone and style of the original text."),
        ("user", "{text}"),
    ]
)
# using the template
translation_prompt = translation_template.format(text=text, source_language=source_language, target_language=target_language)
translation_prompt

# using the model
response = model.invoke(translation_prompt)
response.content



'मुझे पाइथन में प्रोग्रामिंग करना बहुत पसंद है।'

In [17]:
# Building your first chain
from langchain_core.output_parsers import StrOutputParser
# StrOutputParser extracts the plain text string from the raw message object returned by the Language Model.
from langchain_core.runnables import RunnablePassthrough,RunnableLambda
#RunnableLambda is a wrapper that lets you plug any standard Python function into a LangChain pipeline using the pipe (|) operator.

def create_story_chain():
    story_prompt = ChatPromptTemplate.from_messages(
        [
            ("system", "You are a creative storyteller. Write a short and engaging story based on the given theme"),
            ("user", "{theme}\n Main Character: {main_character} \n Setting: {setting}"),
        ]
    )
    # Template for story analysis
    analysis_prompt = ChatPromptTemplate.from_messages(
        [
            ("system", "You are a literacy critic. Analyze the follwing story and provide insights."),
            ("user", "{story}"),
        ]
    )

    story_chain = story_prompt | model | StrOutputParser()

    # function to pass to analysis

    def analyze_story(story_text):
        return {"story": story_text}

    analysis_chain = (
        {"story": story_chain}
        | RunnableLambda(analyze_story) 
        |analysis_prompt 
        | model 
        | StrOutputParser()
    )
    return analysis_chain
    
    

In [18]:
chain = create_story_chain()
chain

{
  story: ChatPromptTemplate(input_variables=['main_character', 'setting', 'theme'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a creative storyteller. Write a short and engaging story based on the given theme'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['main_character', 'setting', 'theme'], input_types={}, partial_variables={}, template='{theme}\n Main Character: {main_character} \n Setting: {setting}'), additional_kwargs={})])
         | ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.9', 'langchain': '1.4.4'}}, client=<groq.resources.chat.completions.Completions object at 0x107caead0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x107cbc710>, model_name='qwen/qwen3.8-27b', model_kwargs={}, groq_api_key=SecretStr('**********'))
         | StrOutputParser()
}
| Runnable

In [19]:
result = chain.invoke({"theme": "A young girl who dreams of becoming a doctor", 
"main_character": "A young girl named Maya", 
"setting": "A small village in India"})
print("Story and Analysis:")
print(result)


Story and Analysis:
As a literacy critic, I have analyzed the short story *"To Cure"* for its narrative structure, character development, thematic resonance, and stylistic elements. Below is a detailed breakdown of the text’s strengths and areas for potential refinement.

### **1. Characterization and Voice**
**Strengths:**
*   **Protagonist as Intellectual Outlier:** Maya is effectively established as an intellectual outlier within her community. The contrast between her engagement with *Gray’s Anatomy* and the other children playing *gilli-danda* immediately sets her apart. This is a classic but effective literary trope that allows the reader to sympathize with her isolation and dedication.
*   **Agency:** Maya is not a passive recipient of the medical crisis; she is the active problem-solver. Her decision to intervene, despite her youth and lack of formal training, drives the plot.
*   **Sensory Detail:** The character is grounded in sensory details—her "chappals," the "tattered boo